# Algorithm 3 — Edit Survival Analysis

**File:** `src/CopilotScope.Collector/Quality/Analyzers.cs` (lines 1–47)

Measures how much of the AI-generated code **survives** after the user's observation window closes.
Two independent telemetry signals are combined into a single survival score.

---

## 1  Input Signals

| Signal | Symbol | Source | Semantics |
|---|---|---|---|
| Four-gram overlap | $F$ | Copilot editor telemetry | Fraction of 4-gram shingles from generated code still present verbatim |
| No-revert | $R$ | Copilot editor telemetry | 1 if the edit was never undone, 0 otherwise (averaged over edits) |

---

## 2  Combined Survival Score

When both signals are present:

$$
\boxed{S = 0.4 \cdot \bar{F} + 0.6 \cdot \bar{R}}
$$

where $\bar{F}$ and $\bar{R}$ are the **per-session means** of each signal over all edits.

When only one signal is available, $S = \bar{F}$ or $S = \bar{R}$ respectively.

**Rationale for $w_R > w_F$:** code can be legitimately *refactored* (low four-gram overlap) while
the semantic intent survives.  Not reverting is the stronger durability signal.

---

## 3  Interpretation Thresholds

| $S$ | Interpretation |
|---|---|
| $S \geq 0.8$ | Code is durable — vast majority survives untouched |
| $0.5 \leq S < 0.8$ | Partial rework — suggestions land but get meaningfully edited |
| $S < 0.5$ | Heavy rewrite — most generated code is replaced or reverted |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def edit_survival(four_gram_samples: list[float], no_revert_samples: list[float]) -> dict:
    fg = np.mean(four_gram_samples) if four_gram_samples else None
    nr = np.mean(no_revert_samples) if no_revert_samples else None

    if fg is not None and nr is not None:
        combined = 0.4 * fg + 0.6 * nr
    elif fg is not None:
        combined = fg
    elif nr is not None:
        combined = nr
    else:
        combined = None

    def interpret(s):
        if s is None: return 'no-data'
        if s >= 0.8: return 'durable'
        if s >= 0.5: return 'partial rework'
        return 'heavy rewrite'

    return {'four_gram': fg, 'no_revert': nr, 'combined': combined,
            'interpretation': interpret(combined)}

# --- Unit results ---
cases = [
    ('Durable — clean acceptance',        [0.90, 0.85, 0.92], [1.0, 1.0, 1.0, 0.9]),
    ('Refactored — low 4-gram, kept',     [0.20, 0.25, 0.30], [0.95, 1.0, 0.9]),
    ('Mixed signals',                     [0.60, 0.55],       [0.70, 0.60, 0.65]),
    ('Mostly reverted',                   [0.30, 0.25],       [0.10, 0.20, 0.15]),
    ('Only 4-gram available',             [0.80, 0.75],       []),
    ('Only no-revert available',          [],                  [0.85, 0.90, 0.80]),
]

print(f'{"Scenario":<38} {"F̄":>7} {"R̄":>9} {"S":>9}  Interpretation')
print('-' * 85)
for name, fg, nr in cases:
    r = edit_survival(fg, nr)
    fg_s  = f"{r['four_gram']:.4f}" if r['four_gram']  is not None else '     —'
    nr_s  = f"{r['no_revert']:.4f}" if r['no_revert']  is not None else '     —'
    comb  = f"{r['combined']:.4f}"  if r['combined']   is not None else '     —'
    print(f'{name:<38} {fg_s:>7} {nr_s:>9} {comb:>9}  {r["interpretation"]}')

In [ ]:
# Iso-score contours: combined S as function of (F, R)
F = np.linspace(0, 1, 200)
R = np.linspace(0, 1, 200)
FF, RR = np.meshgrid(F, R)
SS = 0.4 * FF + 0.6 * RR

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(FF, RR, SS, levels=20, cmap='RdYlGn')
ct = ax.contour(FF, RR, SS, levels=[0.5, 0.8], colors=['orange', 'green'], linewidths=2)
ax.clabel(ct, fmt={0.5: 'S=0.50', 0.8: 'S=0.80'}, fontsize=11)
plt.colorbar(cs, ax=ax, label='Combined survival score S')
ax.set_xlabel('Four-gram survival $\\bar{F}$')
ax.set_ylabel('No-revert survival $\\bar{R}$')
ax.set_title('Edit Survival: $S = 0.4\\bar{F} + 0.6\\bar{R}$')
plt.tight_layout()
plt.savefig('edit_survival_contour.png', dpi=150)
plt.show()

---

## 4  Four-gram Shingling — Exact Mechanics

The **four-gram overlap** $F$ measures textual similarity at the character level.
Given a generated code string $g$ and its surviving form $s$, we extract
all consecutive character subsequences of length 4 (called *4-grams* or *character shingles*):

$$
\mathcal{G}_4(t) = \{ t[i : i+4] \mid 0 \leq i \leq |t| - 4 \}
$$

The overlap fraction for a single edit is then:

$$
F = \frac{|\mathcal{G}_4(g) \cap \mathcal{G}_4(s)|}{|\mathcal{G}_4(g)|}
$$

> **Why sets, not multisets?**  
> Using *sets* avoids double-counting repeated patterns (e.g. a loop body `\n    `)
> and keeps $F \in [0, 1]$ regardless of code length or structure.
> A multiset version would be biased toward longer, more repetitive generated text.

The per-session mean $\bar{F} = \frac{1}{n}\sum_{k=1}^n F_k$ averages over all $n$ edits.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 4: Four-gram shingling — exact mechanics with worked mini-examples
# ─────────────────────────────────────────────────────────────────────────────

def compute_4grams(text: str) -> set[str]:
    """Set of character-level 4-grams from `text`."""
    return {text[i:i+4] for i in range(len(text) - 3)} if len(text) >= 4 else set()

def four_gram_overlap(generated: str, surviving: str) -> float:
    """F = |4grams(generated) ∩ 4grams(surviving)| / |4grams(generated)|."""
    gen_g = compute_4grams(generated)
    if not gen_g:
        return 1.0   # degenerate: nothing to measure
    return len(gen_g & compute_4grams(surviving)) / len(gen_g)

# ── Mini-example A: single-character change (return value 1 → 2) ────────────
gen_A = "    return x + 1"
sur_A = "    return x + 2"

g_A   = compute_4grams(gen_A)
s_A   = compute_4grams(sur_A)
inter_A = g_A & s_A
lost_A  = g_A - s_A

print("Mini-example A: single-character change (return value 1 → 2)")
print(f"  Generated : {repr(gen_A)}")
print(f"  Surviving : {repr(sur_A)}")
print()
print(f"  |generated 4-grams| = {len(g_A)}")
print(f"  Generated 4-grams  = {sorted(g_A)}")
print()
print(f"  |surviving 4-grams| = {len(s_A)}")
print(f"  Surviving  4-grams  = {sorted(s_A)}")
print()
print(f"  Intersection ({len(inter_A)} grams): {sorted(inter_A)}")
print(f"  Lost grams  ({len(lost_A)} grams): {sorted(lost_A)}")
print()
print(f"  F_A = {len(inter_A)} / {len(g_A)} = {len(inter_A)/len(g_A):.4f}")
print()
print("  Interpretation: a 1-char change in a 16-char string loses only the last")
print("  4-gram that contains the changed digit — all prefix grams survive.")

# ── Mini-example B: substantial semantic change ─────────────────────────────
print()
print("─" * 70)
gen_B = "if x > 0:\n    return True"
sur_B = "if x >= 0:\n    return x != 0"

g_B   = compute_4grams(gen_B)
s_B   = compute_4grams(sur_B)
inter_B = g_B & s_B
lost_B  = g_B - s_B

print()
print("Mini-example B: semantic rewrite (different condition and return expression)")
print(f"  Generated : {repr(gen_B)}")
print(f"  Surviving : {repr(sur_B)}")
print()
print(f"  |generated 4-grams| = {len(g_B)}")
print(f"  Generated 4-grams  = {sorted(g_B)}")
print()
print(f"  |surviving 4-grams| = {len(s_B)}")
print(f"  Surviving  4-grams  = {sorted(s_B)}")
print()
print(f"  Intersection ({len(inter_B)} grams): {sorted(inter_B)}")
print(f"  Lost grams  ({len(lost_B)} grams): {sorted(lost_B)}")
print()
print(f"  F_B = {len(inter_B)} / {len(g_B)} = {len(inter_B)/len(g_B):.4f}")
print()
print("  Interpretation: the keyword 'if x' and indentation shingles survive,")
print("  but the comparison operator, return value expression are new → low F.")

---

## 5  Full Session Walkthrough — Step-by-Step Computation

The following session simulates a realistic Python development scenario with **4 edits**:

| Edit | What happened | Expected signal |
|------|---------------|------------------|
| 1 | Simple utility function, accepted and kept verbatim | High F, R=1 |
| 2 | Multiply function, user appended `* 2` to return | High F (most text survives), R=1 |
| 3 | Condition branch, user rewrote the logic significantly | Low F, R=1 (kept but changed) |
| 4 | Placeholder comment, user reverted entirely | F=0, R=0 |

Each row computes $F_k$ via four-gram overlap and $R_k \in \{0, 1\}$ from revert status.
The session score is:
$$
\bar{F} = \frac{1}{4}\sum F_k, \quad \bar{R} = \frac{1}{4}\sum R_k, \quad S = 0.4\bar{F} + 0.6\bar{R}
$$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 5: Full session walkthrough — 4 edits, every value shown
# ─────────────────────────────────────────────────────────────────────────────
from dataclasses import dataclass

@dataclass
class EditEvent:
    label:    str
    generated: str
    surviving: str   # empty string means reverted
    reverted:  bool

session = [
    EditEvent(
        "#1  Simple add() — kept verbatim",
        generated="def add(a, b):\n    return a + b",
        surviving="def add(a, b):\n    return a + b",
        reverted=False,
    ),
    EditEvent(
        "#2  multiply() — user appended '* 2'",
        generated="def multiply(x, y):\n    return x * y",
        surviving="def multiply(x, y):\n    return x * y * 2",
        reverted=False,
    ),
    EditEvent(
        "#3  Condition — logic rewritten by user",
        generated="if x > 0:\n    return True",
        surviving="if x >= 0:\n    return x != 0",
        reverted=False,
    ),
    EditEvent(
        "#4  Placeholder comment — reverted entirely",
        generated="# TODO: implement later\npass",
        surviving="",
        reverted=True,
    ),
]

divider = "=" * 80
thin    = "─" * 80

print(divider)
print("FULL SESSION WALKTHROUGH — EDIT SURVIVAL ANALYSIS (4 edits)")
print(divider)

F_values = []
R_values = []

for e in session:
    print()
    print(thin)
    print(f"  {e.label}")
    print(thin)
    print(f"  Generated : {repr(e.generated)}")
    print(f"  Surviving : {repr(e.surviving) if e.surviving else '(reverted — empty)'}")
    print()

    # ── Four-gram overlap ──────────────────────────────────────────────────
    if e.reverted or not e.surviving:
        f = 0.0
        print("  Four-gram F:")
        print("    Code was reverted — no surviving text → F = 0.0000")
    else:
        gen_g = compute_4grams(e.generated)
        sur_g = compute_4grams(e.surviving)
        inter = gen_g & sur_g
        f = len(inter) / len(gen_g)
        print("  Four-gram F:")
        print(f"    |generated 4-grams| = {len(gen_g)}  →  {sorted(gen_g)}")
        print(f"    |surviving 4-grams| = {len(sur_g)}  →  {sorted(sur_g)}")
        print(f"    intersection        = {len(inter)}  →  {sorted(inter)}")
        print(f"    F = {len(inter)} / {len(gen_g)} = {f:.4f}")

    # ── No-revert ──────────────────────────────────────────────────────────
    r = 0.0 if e.reverted else 1.0
    print(f"  No-revert  R = {r:.1f}  ({'reverted' if e.reverted else 'kept (not undone)'})")

    F_values.append(f)
    R_values.append(r)

# ── Session aggregation ───────────────────────────────────────────────────
print()
print(divider)
print("SESSION AGGREGATION")
print(divider)

F_bar = np.mean(F_values)
R_bar = np.mean(R_values)
S     = 0.4 * F_bar + 0.6 * R_bar

print()
print(f"  Per-edit F values : [{', '.join(f'{v:.4f}' for v in F_values)}]")
print(f"  Per-edit R values : [{', '.join(f'{v:.1f}' for v in R_values)}]")
print()
print(f"  F̄ = ({' + '.join(f'{v:.4f}' for v in F_values)}) / {len(F_values)}")
print(f"    = {sum(F_values):.4f} / {len(F_values)} = {F_bar:.4f}")
print()
print(f"  R̄ = ({' + '.join(f'{v:.1f}' for v in R_values)}) / {len(R_values)}")
print(f"    = {sum(R_values):.1f} / {len(R_values)} = {R_bar:.4f}")
print()
print(f"  S = 0.4 × F̄ + 0.6 × R̄")
print(f"    = 0.4 × {F_bar:.4f} + 0.6 × {R_bar:.4f}")
print(f"    = {0.4*F_bar:.4f} + {0.6*R_bar:.4f}")
print(f"    = {S:.4f}")
print()
label = 'durable' if S >= 0.8 else 'partial rework' if S >= 0.5 else 'heavy rewrite'
print(f"  Interpretation: {label}")
print()
print("  Note: Even with one full revert (R=0) and one low-F rewrite (F≈0.31),")
print("  the session scores 'partial rework' because two high-quality edits")
print("  and three no-revert outcomes pull the averages upward.")

In [ ]:
# Per-edit breakdown visualisation
labels = ['Edit 1\n(verbatim)', 'Edit 2\n(append)', 'Edit 3\n(rewrite)', 'Edit 4\n(reverted)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors_F = ['#2ecc71' if f >= 0.8 else '#f39c12' if f >= 0.5 else '#e74c3c' for f in F_values]
colors_R = ['#2ecc71' if r == 1.0 else '#e74c3c' for r in R_values]

# Left: per-edit F and R side by side
x = np.arange(len(labels))
w = 0.35
axes[0].bar(x - w/2, F_values, w, label='Four-gram F', color=colors_F, edgecolor='white')
axes[0].bar(x + w/2, R_values, w, label='No-revert R', color=colors_R, alpha=0.75, edgecolor='white')
axes[0].axhline(F_bar, color='steelblue', linestyle='--', linewidth=1.5, label=f'F̄={F_bar:.3f}')
axes[0].axhline(R_bar, color='darkorange', linestyle='--', linewidth=1.5, label=f'R̄={R_bar:.3f}')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, fontsize=9)
axes[0].set_ylim(0, 1.1); axes[0].set_ylabel('Signal value')
axes[0].set_title('Per-edit signals F and R')
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)
for i, (f, r) in enumerate(zip(F_values, R_values)):
    axes[0].text(i - w/2, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)
    axes[0].text(i + w/2, r + 0.02, f'{r:.1f}',  ha='center', fontsize=8)

# Right: contribution breakdown to S
contrib_F = 0.4 * F_bar
contrib_R = 0.6 * R_bar
axes[1].bar(['0.4·F̄', '0.6·R̄', 'S (total)'],
            [contrib_F, contrib_R, S],
            color=['steelblue', 'darkorange', '#8e44ad'], edgecolor='white')
axes[1].axhline(0.5, color='orange', linestyle='--', linewidth=1.5, label='partial rework / heavy rewrite')
axes[1].axhline(0.8, color='green',  linestyle='--', linewidth=1.5, label='durable threshold')
for i, v in enumerate([contrib_F, contrib_R, S]):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('Score contribution')
axes[1].set_title(f'Score decomposition  S = {S:.4f}')
axes[1].legend(fontsize=8); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Edit Survival — full session breakdown', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('edit_survival_session_breakdown.png', dpi=150)
plt.show()

---

## 6  Weight Sensitivity Analysis

The formula $S = w_F \bar{F} + w_R \bar{R}$ with $w_F + w_R = 1$ has one free parameter.
The table below evaluates six weight configurations across four representative
$(\bar{F}, \bar{R})$ pairs, showing how the chosen 0.4/0.6 split compares to alternatives.

**Key insight — the refactored scenario** $(\bar{F}=0.25, \bar{R}=0.95)$:
a developer refactors AI-generated code (low textual overlap) but never reverts it
(high no-revert).  This is a *semantic acceptance* — the AI's intent survived even
though the exact text didn't.  A four-gram-dominant weighting would undervalue this
scenario significantly, misclassifying it as nearly "heavy rewrite".

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 6: Weight sensitivity — show S under 6 (w_F, w_R) combinations
# ─────────────────────────────────────────────────────────────────────────────

weight_configs = [
    (0.2, 0.8, "no-revert extreme"),
    (0.3, 0.7, "no-revert heavy"),
    (0.4, 0.6, "default ★"),
    (0.5, 0.5, "equal"),
    (0.6, 0.4, "4-gram heavy"),
    (0.8, 0.2, "4-gram extreme"),
]

# Representative (F̄, R̄) pairs drawn from earlier scenarios
scenarios = [
    ("Durable (verbatim)",          0.8900, 0.9750),
    ("Refactored (semantic keep)",  0.2500, 0.9500),
    ("Mixed signals",               0.5750, 0.6500),
    ("Mostly reverted",             0.2750, 0.1500),
    ("Session walkthrough (§5)",    F_bar,  R_bar),
]

# Header
col_w = 10
header = f"{'Scenario':<30}  {'F̄':>6}  {'R̄':>6}"
for wf, wr, label in weight_configs:
    header += f"  {label[:14]:>14}"
print(header)
print("-" * (30 + 2 + 6 + 2 + 6 + len(weight_configs) * 16))

for name, Fb, Rb in scenarios:
    row = f"{name:<30}  {Fb:>6.4f}  {Rb:>6.4f}"
    for wf, wr, label in weight_configs:
        s = wf * Fb + wr * Rb
        flag = " ★" if label == "default ★" else "  "
        interp = "dur" if s >= 0.8 else "par" if s >= 0.5 else "HRW"
        row += f"  {s:.4f}{flag}({interp})"
    print(row)

print()
print("Legend: dur = durable, par = partial rework, HRW = heavy rewrite, ★ = CopilotScope default")
print()
print("Critical observation: 'Refactored (semantic keep)' scenario")
for wf, wr, label in weight_configs:
    s = wf * 0.25 + wr * 0.95
    interp = "durable" if s >= 0.8 else "partial rework" if s >= 0.5 else "HEAVY REWRITE"
    print(f"  w_F={wf:.1f} / w_R={wr:.1f}  →  S = {s:.4f}  ({interp})")

In [ ]:
# Sensitivity heat-map: S(F̄, R̄) for three weight configurations side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs_plot = [
    (0.6, 0.4, '4-gram heavy (w_F=0.6)'),
    (0.4, 0.6, 'Default (w_F=0.4) ★'),
    (0.2, 0.8, 'No-revert heavy (w_F=0.2)'),
]

# The four scenarios as annotated points
pts = [
    (0.89, 0.975, 'Durable'),
    (0.25, 0.95,  'Refactored'),
    (0.575, 0.65, 'Mixed'),
    (0.275, 0.15, 'Reverted'),
]

for ax, (wf, wr, title) in zip(axes, configs_plot):
    SS = wf * FF + wr * RR
    cs = ax.contourf(FF, RR, SS, levels=20, cmap='RdYlGn', vmin=0, vmax=1)
    ct = ax.contour(FF, RR, SS, levels=[0.5, 0.8], colors=['orange', 'green'], linewidths=1.5)
    ax.clabel(ct, fmt={0.5: 'S=0.50', 0.8: 'S=0.80'}, fontsize=9)
    for px, py, pname in pts:
        sv = wf * px + wr * py
        ax.scatter(px, py, s=80, color='black', zorder=5)
        ax.annotate(f'{pname}\n{sv:.2f}', (px, py),
                    textcoords='offset points', xytext=(6, 4), fontsize=7,
                    color='black',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_xlabel('$\\bar{F}$ (four-gram mean)', fontsize=9)
    ax.set_ylabel('$\\bar{R}$ (no-revert mean)', fontsize=9)
    ax.set_title(title, fontsize=10)

fig.colorbar(cs, ax=axes.ravel().tolist(), label='Combined survival score S', shrink=0.8)
plt.suptitle('Weight Sensitivity: S = w_F·F̄ + w_R·R̄ across three configurations',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('edit_survival_weight_sensitivity.png', dpi=150)
plt.show()

---

## 7  Connection to the Acceptance Component

The survival score $\bar{S}$ (session mean combined score) feeds directly into the
**Acceptance** component of the Composite Quality Engine:

$$
v_A = 0.6\,\alpha + 0.4\,\bar{S}
$$

where $\alpha = a / (a + j)$ is the acceptance ratio (accepted over total edits).

This two-level design captures **two different moments**:

| Level | Moment | What it measures |
|---|---|---|
| $\alpha$ | At suggestion time | Did the user accept or reject the proposal? |
| $\bar{S}$ | After observation window | Did the accepted code survive, or was it heavily rewritten / reverted? |

A high acceptance rate with low survival ($\alpha=0.90$, $\bar{S}=0.25$) means
the user clicks Accept but then immediately rewrites or reverts — a **false positive**
acceptance metric.  The survival term penalises exactly this pattern.

Conversely, a lower acceptance rate with high survival ($\alpha=0.60$, $\bar{S}=0.90$)
means the user is selective but keeps what they do accept — **high-quality selection**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 7: Connection to acceptance component — v_A = 0.6α + 0.4S̄
# ─────────────────────────────────────────────────────────────────────────────

acceptance_examples = [
    {"label": "High accept + high survival",   "alpha": 0.90, "s_bar": 0.85},
    {"label": "High accept + heavy rewrite",   "alpha": 0.90, "s_bar": 0.25},
    {"label": "Moderate accept + high survival","alpha": 0.60, "s_bar": 0.90},
    {"label": "Low accept + durable keeps",    "alpha": 0.40, "s_bar": 0.95},
    {"label": "Low accept + low survival",     "alpha": 0.40, "s_bar": 0.20},
    {"label": "Session walkthrough (§5)",      "alpha": 0.75, "s_bar": S},
]

print(f"{'Scenario':<35}  {'α':>6}  {'S̄':>7}  {'v_A=0.6α+0.4S̄':>16}  Breakdown  Quality")
print("-" * 90)
for ex in acceptance_examples:
    a     = ex["alpha"]
    s_bar = ex["s_bar"]
    v_a   = 0.6 * a + 0.4 * s_bar
    quality = "excellent" if v_a >= 0.8 else "good" if v_a >= 0.65 else "moderate" if v_a >= 0.5 else "poor"
    print(f"{ex['label']:<35}  {a:>6.2f}  {s_bar:>7.4f}  {v_a:>16.4f}  "
          f"0.6×{a:.2f}+0.4×{s_bar:.4f}  {quality}")

print()
print("Key observation: 'High accept + heavy rewrite' scores LOWER than")
print("'Moderate accept + high survival' (0.6400 < 0.7200).")
print("The survival term correctly penalises acceptance without durability.")

In [ ]:
# v_A surface: acceptance component as function of (α, S̄)
alpha_range = np.linspace(0, 1, 200)
s_range     = np.linspace(0, 1, 200)
AA, SS2     = np.meshgrid(alpha_range, s_range)
VA          = 0.6 * AA + 0.4 * SS2

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(AA, SS2, VA, levels=20, cmap='RdYlGn')
ct = ax.contour(AA, SS2, VA, levels=[0.5, 0.65, 0.80], colors=['red', 'orange', 'green'], linewidths=1.8)
ax.clabel(ct, fmt={0.5: 'v_A=0.50', 0.65: 'v_A=0.65', 0.80: 'v_A=0.80'}, fontsize=10)
plt.colorbar(cs, ax=ax, label='Acceptance component v_A')

# Annotate the example points
pts_va = [
    (0.90, 0.85, 'Hi-acc Hi-surv'),
    (0.90, 0.25, 'Hi-acc Lo-surv'),
    (0.60, 0.90, 'Mod-acc Hi-surv'),
    (0.40, 0.95, 'Lo-acc Hi-surv'),
    (0.40, 0.20, 'Lo-acc Lo-surv'),
]
for ax_pt, ay_pt, name in pts_va:
    v = 0.6 * ax_pt + 0.4 * ay_pt
    ax.scatter(ax_pt, ay_pt, s=90, color='navy', zorder=5)
    ax.annotate(f'{name}\nv_A={v:.3f}', (ax_pt, ay_pt),
                textcoords='offset points', xytext=(8, 4), fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.85))

ax.set_xlabel('Acceptance ratio α = a/(a+j)')
ax.set_ylabel('Mean survival score S̄')
ax.set_title('Acceptance component $v_A = 0.6\\alpha + 0.4\\bar{S}$')
plt.tight_layout()
plt.savefig('edit_survival_acceptance_surface.png', dpi=150)
plt.show()

---

## 8  Edge Cases and Fallback Behaviour

| Situation | Available signals | Formula applied | Note |
|---|---|---|---|
| Both signals | $\bar{F}$ and $\bar{R}$ | $S = 0.4\bar{F}+0.6\bar{R}$ | normal path |
| Only four-gram | $\bar{F}$ only | $S = \bar{F}$ | no weight adjustment; $S$ is unweighted |
| Only no-revert | $\bar{R}$ only | $S = \bar{R}$ | no weight adjustment |
| No data | neither | `no-data` status | score field is null; not fed to composite |
| All edits reverted | $R_k=0$ ∀k, $F_k=0$ ∀k | $S = 0.0$ | heavy rewrite |
| Single edit, kept verbatim | $F=1.0$, $R=1.0$ | $S = 0.4+0.6 = 1.0$ | perfect |

The **fallback without weight adjustment** is a deliberate design choice:
if only one signal is present, we do not pretend to know the other signal's value.
A neutral prior (e.g. 0.5) would artificially compress scores; the single-signal
value is the best unbiased estimate available.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 8: Edge cases — exhaustive enumeration with values
# ─────────────────────────────────────────────────────────────────────────────

edge_cases = [
    ("Both present — durable",            [0.90, 0.85, 0.92], [1.0, 1.0, 1.0, 0.9]),
    ("Both present — heavy rewrite",      [0.10, 0.05],       [0.0, 0.0, 0.0]),
    ("Only four-gram (0.78)",             [0.80, 0.75],       []),
    ("Only no-revert (0.85)",             [],                 [0.85, 0.90, 0.80]),
    ("Single edit kept verbatim",         [1.0],              [1.0]),
    ("Single edit fully reverted",        [0.0],              [0.0]),
    ("All edits reverted",                [0.0, 0.0, 0.0],    [0.0, 0.0, 0.0]),
    ("No data at all",                    [],                 []),
]

print(f"{'Scenario':<40} {'F̄':>8} {'R̄':>8} {'S':>8}  {'Status':<12}  Note")
print("-" * 110)
for name, fg_list, nr_list in edge_cases:
    r = edit_survival(fg_list, nr_list)
    fg_s = f"{r['four_gram']:.4f}" if r['four_gram'] is not None else "  null"
    nr_s = f"{r['no_revert']:.4f}" if r['no_revert'] is not None else "  null"
    s_s  = f"{r['combined']:.4f}"  if r['combined']  is not None else "  null"
    if r['combined'] is None:
        status = "no-data"
        note   = "score=null, not fed to composite"
    elif not fg_list:
        status = "ok (nr only)"
        note   = f"S = R̄ = {r['combined']:.4f}, no 4-gram weighting"
    elif not nr_list:
        status = "ok (fg only)"
        note   = f"S = F̄ = {r['combined']:.4f}, no no-revert weighting"
    else:
        status = "ok"
        note   = r['interpretation']
    print(f"{name:<40} {fg_s:>8} {nr_s:>8} {s_s:>8}  {status:<12}  {note}")

In [ ]:
# Summary visualisation: all named scenarios in a single ranked bar chart

all_scenarios = [
    ('Durable — clean',          edit_survival([0.90, 0.85, 0.92], [1.0, 1.0, 1.0, 0.9])),
    ('Session walkthrough §5',   {'combined': S, 'four_gram': F_bar, 'no_revert': R_bar,
                                  'interpretation': 'partial rework'}),
    ('Refactored — kept',        edit_survival([0.20, 0.25, 0.30], [0.95, 1.0, 0.9])),
    ('Mixed signals',            edit_survival([0.60, 0.55],       [0.70, 0.60, 0.65])),
    ('Only 4-gram (0.78)',        edit_survival([0.80, 0.75],       [])),
    ('Only no-revert (0.85)',     edit_survival([],                 [0.85, 0.90, 0.80])),
    ('Mostly reverted',          edit_survival([0.30, 0.25],       [0.10, 0.20, 0.15])),
    ('All reverted (worst)',      edit_survival([0.0, 0.0],         [0.0, 0.0])),
]

names  = [n for n, _ in all_scenarios]
scores = [d['combined'] for _, d in all_scenarios]
colors = ['#2ecc71' if s >= 0.8 else '#f39c12' if s >= 0.5 else '#e74c3c'
          for s in scores]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, scores, color=colors, edgecolor='white')
ax.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='partial/heavy threshold (0.5)')
ax.axvline(0.8, color='green',  linestyle='--', linewidth=1.5, label='durable threshold (0.8)')
for bar, s in zip(bars, scores):
    ax.text(s + 0.01, bar.get_y() + bar.get_height()/2,
            f'{s:.4f}', va='center', fontsize=9)
ax.set_xlim(0, 1.12)
ax.set_xlabel('Combined survival score S')
ax.set_title('Edit Survival Scores — all scenarios ranked')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('edit_survival_all_scenarios.png', dpi=150)
plt.show()

print()
print("Colour key:  green = durable (S≥0.8),  orange = partial rework (0.5≤S<0.8),  red = heavy rewrite (S<0.5)")